Set up up the Spark master
  - "local" for local execution
  - "local[*]" for local execution using all core
  - "spark://spark-master:7077" to connect to the Spark master running on the docker container

For the first two options you will to set up a Python environment and install pyspark.

In [2]:
master="spark://spark-master:7077"
filename = '/data/lab01/flights.csv'

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
             .master(master) \
             .appName('flights') \
             .getOrCreate()

df = spark.read.option("delimiter", ",").option("header", True).csv(filename)
df.show()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 08:30:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/24 08:30:23 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
[Stage 1:>                                                          (0 + 1) / 1]

+------------+-----------+-------+-------+-----+------+------+-------+----+------------------+--------+---------------+------------------+--------+-------------+------------+--------+
|day of Month|day of Week|carrier|tailnum|flnum|org_id|origin|dest_id|dest|scheduled dep time|dep time|departure delay|scheduled arr time|arr time|arrival delay|elapsed time|distance|
+------------+-----------+-------+-------+-----+------+------+-------+----+------------------+--------+---------------+------------------+--------+-------------+------------+--------+
|           1|          3|     AA| N338AA|    1| 12478|   JFK|  12892| LAX|               900|     914|             14|              1225|    1238|           13|         385|    2475|
|           2|          4|     AA| N338AA|    1| 12478|   JFK|  12892| LAX|               900|     857|              0|              1225|    1226|            1|         385|    2475|
|           4|          6|     AA| N327AA|    1| 12478|   JFK|  12892| LAX|     

## Flights per airport

In [14]:
result = df.groupBy("origin").count()
result.show()

+------+-----+
|origin|count|
+------+-----+
|   MSY| 3130|
|   GEG|  699|
|   SNA| 3159|
|   BUR| 1683|
|   GTF|  149|
|   GRR|  720|
|   EUG|  416|
|   GSO|  530|
|   PVD|  813|
|   MYR|  106|
|   OAK| 3319|
|   MSN|  696|
|   FAR|  368|
|   DCA| 5336|
|   CID|  396|
|   LEX|  357|
|   ORF|  884|
|   CRW|  205|
|   SAV|  451|
|   TRI|  159|
+------+-----+
only showing top 20 rows


In [15]:
result = df.groupBy("dest").count()
result.show()

+----+-----+
|dest|count|
+----+-----+
| MSY| 3161|
| GEG|  699|
| SNA| 3159|
| BUR| 1687|
| GTF|  149|
| GRB|  263|
| IDA|  214|
| GRR|  736|
| JLN|   61|
| EUG|  422|
| PVD|  826|
| GSO|  537|
| OAK| 3310|
| FAR|  368|
| MSN|  712|
| COD|   61|
| DCA| 5350|
| CID|  403|
| MLU|  260|
| LWS|   53|
+----+-----+
only showing top 20 rows


In [16]:
ordered = result.orderBy("count", ascending=False)
ordered.show()

+----+-----+
|dest|count|
+----+-----+
| ATL|28279|
| DFW|22724|
| LAX|18014|
| ORD|17874|
| DEN|17175|
| IAH|13332|
| SFO|13101|
| PHX|13092|
| LAS|10841|
| CLT| 9358|
| MCO| 8793|
| SLC| 8520|
| EWR| 8153|
| MSP| 7728|
| SEA| 7641|
| BOS| 7584|
| LGA| 7527|
| MIA| 7184|
| JFK| 7116|
| DTW| 6963|
+----+-----+
only showing top 20 rows


In [17]:
result = df.groupBy("origin", "dest").count()
result.show()

+------+----+-----+
|origin|dest|count|
+------+----+-----+
|   ATL| GSP|  165|
|   SNA| PHX|  352|
|   PHL| MCO|  380|
|   PBI| DCA|   45|
|   EWR| STT|   31|
|   ORD| PDX|  180|
|   LAS| LIT|   31|
|   MCI| MKE|   54|
|   MDW| MEM|   54|
|   SMF| BUR|  188|
|   CAE| ATL|  214|
|   TPA| CVG|   35|
|   RDU| SLC|   21|
|   CPR| DEN|  110|
|   PSP| JFK|    4|
|   EYW| TPA|   31|
|   AUS| ELP|   61|
|   SJC| ONT|  106|
|   MSP| BOI|   28|
|   ATL| HDN|   26|
+------+----+-----+
only showing top 20 rows


In [18]:
ordered = result.orderBy("count", ascending=False)
ordered.show()

+------+----+-----+
|origin|dest|count|
+------+----+-----+
|   SFO| LAX| 1080|
|   LAX| SFO| 1074|
|   LAS| LAX| 1025|
|   LAX| LAS| 1021|
|   JFK| LAX|  868|
|   LAX| JFK|  861|
|   HNL| OGG|  820|
|   OGG| HNL|  820|
|   LAX| PHX|  802|
|   PHX| LAX|  798|
|   ATL| LGA|  746|
|   SJC| LAX|  745|
|   LGA| ATL|  744|
|   LAX| SJC|  742|
|   LAX| SAN|  728|
|   SAN| LAX|  720|
|   LAX| DFW|  675|
|   DFW| LAX|  674|
|   MCO| ATL|  673|
|   ATL| MCO|  672|
+------+----+-----+
only showing top 20 rows


In [ ]:
result = df.groupBy("origin", "dest").avg("elapsed time")
result.show()

AnalysisException: "elapsed time" is not a numeric column. Aggregation function can only be applied on a numeric column.